# 104 — TCGA Methylation Coverage Assessment

## Objective

Assess the file-, sample-, case-, project-, and platform-level coverage of harmonized TCGA primary-tumor DNA methylation beta-value data before defining the final methylation cohort.

This notebook evaluates the availability of methylation data across array platforms and quantifies its overlap with the frozen TCGA RNA-seq cohort. Its purpose is to support a documented platform-selection strategy while avoiding premature cohort restriction based solely on methylation-platform availability.

## GDC candidate query

The candidate methylation file set was exported from the GDC Data Portal using the following filters:

- Program: TCGA
- Data Category: DNA Methylation
- Data Type: Methylation Beta Value
- Experimental Strategy: Methylation Array
- Workflow Type: SeSAMe Methylation Beta Estimation
- Access: Open
- Tissue Type: Tumor
- Tumor Descriptor: Primary

No platform filter was applied at export time. HM27, HM450, EPIC, and any other represented platforms are retained for explicit coverage assessment.

## Inputs

The all-platform candidate query is represented by:

- `config/manifests/tcga_methylation/gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values.txt`
- `config/manifests/tcga_methylation/gdc_sample_sheet_tcga_primary_tumor_methylation_array_sesame_beta_values.tsv`
- `config/manifests/tcga_methylation/gdc_metadata_tcga_primary_tumor_methylation_array_sesame_beta_values.json`

RNA-seq overlap will be evaluated against the version-controlled frozen TCGA RNA-seq file index generated by notebook 102.

## Analytical questions

This notebook will determine:

- how many methylation files, samples, cases, and TCGA projects are available;
- which methylation-array platforms are represented;
- how platform coverage varies across TCGA projects;
- how many methylation cases overlap the frozen RNA-seq cohort;
- whether platform selection would introduce substantial project-specific attrition;
- the expected download volume associated with each platform strategy.

## Scope

This notebook performs metadata-level coverage assessment only. It does not:

- download or parse methylation beta-value payloads;
- select the final methylation platform strategy in advance;
- freeze the final methylation cohort;
- harmonize probes across array generations;
- select one sample per case;
- construct the final multi-omic cohort;
- perform biological or program-discovery analysis.

The resulting evidence will guide the subsequent TCGA methylation cohort freeze and download-validation notebooks.

In [3]:
# =============================================================================
# 104 — TCGA Methylation Coverage Assessment
# Imports, project-root detection, and input paths
# =============================================================================

import json
import sys
from pathlib import Path

import pandas as pd

# Allow imports from src/ when running this notebook from a notebooks subdirectory.
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "pancancer-epigenetics":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate project root directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import Paths

# Candidate all-platform TCGA methylation query exports.
METHYLATION_MANIFEST_DIR = (
    Paths.config
    / "manifests"
    / "tcga_methylation"
)

METHYLATION_MANIFEST_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values.txt"
)

METHYLATION_SAMPLE_SHEET_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_sample_sheet_tcga_primary_tumor_methylation_array_sesame_beta_values.tsv"
)

METHYLATION_METADATA_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_metadata_tcga_primary_tumor_methylation_array_sesame_beta_values.json"
)

# Frozen TCGA RNA-seq cohort used for cross-modality overlap assessment.
RNA_FILE_INDEX_PATH = (
    Paths.config
    / "manifests"
    / "tcga_rna"
    / "gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv"
)

print(f"Project root:             {PROJECT_ROOT}")
print(f"Methylation manifest:     {METHYLATION_MANIFEST_PATH}")
print(f"Methylation sample sheet: {METHYLATION_SAMPLE_SHEET_PATH}")
print(f"Methylation metadata:     {METHYLATION_METADATA_PATH}")
print(f"RNA-seq file index:       {RNA_FILE_INDEX_PATH}")

Project root:             C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics
Methylation manifest:     C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values.txt
Methylation sample sheet: C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_sample_sheet_tcga_primary_tumor_methylation_array_sesame_beta_values.tsv
Methylation metadata:     C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_metadata_tcga_primary_tumor_methylation_array_sesame_beta_values.json
RNA-seq file index:       C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_rna\gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv


In [4]:
# =============================================================================
# Validate input availability and load tabular inputs
# =============================================================================

required_input_paths = {
    "methylation_manifest": METHYLATION_MANIFEST_PATH,
    "methylation_sample_sheet": METHYLATION_SAMPLE_SHEET_PATH,
    "methylation_metadata": METHYLATION_METADATA_PATH,
    "rna_file_index": RNA_FILE_INDEX_PATH,
}

missing_inputs = [
    path
    for path in required_input_paths.values()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing required coverage-assessment inputs:\n"
        + "\n".join(str(path) for path in missing_inputs)
    )

non_file_inputs = [
    path
    for path in required_input_paths.values()
    if not path.is_file()
]

if non_file_inputs:
    raise ValueError(
        "Expected input paths are not files:\n"
        + "\n".join(str(path) for path in non_file_inputs)
    )

print("Required input files validated:")

for label, path in required_input_paths.items():
    size_mib = path.stat().st_size / 1024**2
    print(f"- {label}: {size_mib:,.3f} MiB")

methylation_manifest_df = pd.read_csv(
    METHYLATION_MANIFEST_PATH,
    sep="\t",
    dtype="string",
)

methylation_sample_sheet_df = pd.read_csv(
    METHYLATION_SAMPLE_SHEET_PATH,
    sep="\t",
    dtype="string",
)

rna_file_index_df = pd.read_csv(
    RNA_FILE_INDEX_PATH,
    sep="\t",
    dtype="string",
)

print("\nTabular inputs loaded.")
print(
    "Methylation manifest shape:    ",
    methylation_manifest_df.shape,
)
print(
    "Methylation sample sheet shape:",
    methylation_sample_sheet_df.shape,
)
print(
    "RNA-seq file index shape:      ",
    rna_file_index_df.shape,
)

print("\nMethylation manifest columns:")
print(methylation_manifest_df.columns.tolist())

print("\nMethylation sample sheet columns:")
print(methylation_sample_sheet_df.columns.tolist())

print("\nRNA-seq file index columns:")
print(rna_file_index_df.columns.tolist())

print("\nMethylation manifest preview:")
display(methylation_manifest_df.head(3))

print("\nMethylation sample sheet preview:")
display(methylation_sample_sheet_df.head(3))

Required input files validated:
- methylation_manifest: 1.729 MiB
- methylation_sample_sheet: 2.359 MiB
- methylation_metadata: 14.067 MiB
- rna_file_index: 2.858 MiB

Tabular inputs loaded.
Methylation manifest shape:     (10950, 5)
Methylation sample sheet shape: (10950, 11)
RNA-seq file index shape:       (10308, 14)

Methylation manifest columns:
['id', 'filename', 'md5', 'size', 'state']

Methylation sample sheet columns:
['File ID', 'File Name', 'Data Category', 'Data Type', 'Project ID', 'Case ID', 'Sample ID', 'Tissue Type', 'Tumor Descriptor', 'Specimen Type', 'Preservation Method']

RNA-seq file index columns:
['file_id', 'file_name', 'md5', 'file_size_bytes', 'state', 'project_id', 'case_id', 'sample_id', 'data_category', 'data_type', 'tissue_type', 'tumor_descriptor', 'specimen_type', 'preservation_method']

Methylation manifest preview:


,id,filename,md5,size,state
0,cf8a9d2e-0967-453d-9de9-8140b1d4afda,a3dad6d1-c405-4458-b5d2-63da84596781.methylati...,8ca7304ae86d70ee30237fe0ffaee2d4,763817,released
1,a099690a-23d9-4b0c-b033-c2eef296bbce,19d539d7-e9ed-42c4-b374-e8cd4798b30e.methylati...,304ba4414f0c377f055d1c5185655545,765130,released
2,7a4a5192-3f01-442e-ae1a-015eff6fb8ac,ee16a10b-11e6-4846-9f8c-59d8d6be9faa.methylati...,9ee96829be340c9e8e807ecfa7c38d4f,773740,released



Methylation sample sheet preview:


,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
0,cf8a9d2e-0967-453d-9de9-8140b1d4afda,a3dad6d1-c405-4458-b5d2-63da84596781.methylati...,DNA Methylation,Methylation Beta Value,TCGA-BRCA,TCGA-BH-A18H,TCGA-BH-A18H-01A,Tumor,Primary,Solid Tissue,OCT
1,a099690a-23d9-4b0c-b033-c2eef296bbce,19d539d7-e9ed-42c4-b374-e8cd4798b30e.methylati...,DNA Methylation,Methylation Beta Value,TCGA-BRCA,TCGA-E2-A14P,TCGA-E2-A14P-01A,Tumor,Primary,Solid Tissue,Unknown
2,7a4a5192-3f01-442e-ae1a-015eff6fb8ac,ee16a10b-11e6-4846-9f8c-59d8d6be9faa.methylati...,DNA Methylation,Methylation Beta Value,TCGA-BRCA,TCGA-AN-A04A,TCGA-AN-A04A-01A,Tumor,Primary,Solid Tissue,OCT


In [6]:
# =============================================================================
# Validate methylation manifest and sample-sheet consistency
# =============================================================================

expected_manifest_columns = {
    "id",
    "filename",
    "md5",
    "size",
    "state",
}

expected_sample_sheet_columns = {
    "File ID",
    "File Name",
    "Data Category",
    "Data Type",
    "Project ID",
    "Case ID",
    "Sample ID",
    "Tissue Type",
    "Tumor Descriptor",
    "Specimen Type",
    "Preservation Method",
}

missing_manifest_columns = (
    expected_manifest_columns
    - set(methylation_manifest_df.columns)
)

missing_sample_sheet_columns = (
    expected_sample_sheet_columns
    - set(methylation_sample_sheet_df.columns)
)

if missing_manifest_columns:
    raise ValueError(
        "Methylation manifest is missing required columns: "
        + ", ".join(sorted(missing_manifest_columns))
    )

if missing_sample_sheet_columns:
    raise ValueError(
        "Methylation sample sheet is missing required columns: "
        + ", ".join(sorted(missing_sample_sheet_columns))
    )

manifest_required_nulls = (
    methylation_manifest_df[
        ["id", "filename", "md5", "size", "state"]
    ]
    .isna()
    .sum()
)

sample_sheet_required_nulls = (
    methylation_sample_sheet_df[
        [
            "File ID",
            "File Name",
            "Data Category",
            "Data Type",
            "Project ID",
            "Case ID",
            "Sample ID",
            "Tissue Type",
            "Tumor Descriptor",
        ]
    ]
    .isna()
    .sum()
)

if (manifest_required_nulls > 0).any():
    raise ValueError(
        "Methylation manifest contains missing required values."
    )

if (sample_sheet_required_nulls > 0).any():
    raise ValueError(
        "Methylation sample sheet contains missing required values."
    )

methylation_manifest_df["size"] = pd.to_numeric(
    methylation_manifest_df["size"],
    errors="raise",
).astype("int64")

if (methylation_manifest_df["size"] <= 0).any():
    raise ValueError(
        "Methylation manifest contains non-positive file sizes."
    )

duplicated_manifest_ids = (
    methylation_manifest_df["id"].duplicated(keep=False)
)

duplicated_sample_sheet_ids = (
    methylation_sample_sheet_df["File ID"].duplicated(keep=False)
)

if duplicated_manifest_ids.any():
    raise ValueError(
        "Duplicated file IDs detected in the methylation manifest."
    )

if duplicated_sample_sheet_ids.any():
    raise ValueError(
        "Duplicated file IDs detected in the methylation sample sheet."
    )

manifest_file_ids = set(methylation_manifest_df["id"])
sample_sheet_file_ids = set(methylation_sample_sheet_df["File ID"])

only_in_manifest = manifest_file_ids - sample_sheet_file_ids
only_in_sample_sheet = sample_sheet_file_ids - manifest_file_ids

if only_in_manifest or only_in_sample_sheet:
    raise ValueError(
        "Methylation manifest and sample sheet do not contain "
        "the same file IDs."
    )

file_name_check_df = methylation_manifest_df[
    ["id", "filename"]
].merge(
    methylation_sample_sheet_df[
        ["File ID", "File Name"]
    ],
    left_on="id",
    right_on="File ID",
    how="inner",
    validate="one_to_one",
)

filename_mismatches = (
    file_name_check_df["filename"]
    != file_name_check_df["File Name"]
)

if filename_mismatches.any():
    raise ValueError(
        "Manifest and sample-sheet filenames do not match."
    )

query_definition_checks = {
    "all_released": (
        methylation_manifest_df["state"] == "released"
    ).all(),
    "all_dna_methylation": (
        methylation_sample_sheet_df["Data Category"]
        == "DNA Methylation"
    ).all(),
    "all_methylation_beta_values": (
        methylation_sample_sheet_df["Data Type"]
        == "Methylation Beta Value"
    ).all(),
    "all_tumor": (
        methylation_sample_sheet_df["Tissue Type"]
        == "Tumor"
    ).all(),
    "all_primary": (
        methylation_sample_sheet_df["Tumor Descriptor"]
        == "Primary"
    ).all(),
}

for check_name, passed in query_definition_checks.items():
    print(f"{check_name}: {passed}")

failed_query_checks = [
    check_name
    for check_name, passed in query_definition_checks.items()
    if not passed
]

if failed_query_checks:
    raise ValueError(
        "Candidate methylation cohort failed query-definition checks: "
        + ", ".join(failed_query_checks)
    )

expected_download_size_gib = (
    methylation_manifest_df["size"].sum() / 1024**3
)

print("\nMethylation tabular exports validated.")
print(f"Files:                  {len(manifest_file_ids):,}")
print(
    "Unique projects:        "
    f"{methylation_sample_sheet_df['Project ID'].nunique():,}"
)
print(
    "Unique cases:           "
    f"{methylation_sample_sheet_df['Case ID'].nunique():,}"
)
print(
    "Unique samples:         "
    f"{methylation_sample_sheet_df['Sample ID'].nunique():,}"
)
print(
    f"Expected download size: {expected_download_size_gib:,.3f} GiB"
)

all_released: True
all_dna_methylation: True
all_methylation_beta_values: True
all_tumor: True
all_primary: True

Methylation tabular exports validated.
Files:                  10,950
Unique projects:        33
Unique cases:           10,610
Unique samples:         10,656
Expected download size: 107.994 GiB


In [7]:
# =============================================================================
# Load and inspect GDC methylation metadata structure
# =============================================================================

with METHYLATION_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    methylation_metadata_json = json.load(handle)

print(
    "Metadata top-level type:",
    type(methylation_metadata_json).__name__,
)

metadata_records = None

if isinstance(methylation_metadata_json, list):
    metadata_records = methylation_metadata_json

elif isinstance(methylation_metadata_json, dict):
    print(
        "Metadata top-level keys:",
        list(methylation_metadata_json.keys()),
    )

    if isinstance(
        methylation_metadata_json.get("hits"),
        list,
    ):
        metadata_records = methylation_metadata_json["hits"]

    elif isinstance(
        methylation_metadata_json.get("data"),
        list,
    ):
        metadata_records = methylation_metadata_json["data"]

    elif (
        isinstance(
            methylation_metadata_json.get("data"),
            dict,
        )
        and isinstance(
            methylation_metadata_json["data"].get("hits"),
            list,
        )
    ):
        metadata_records = (
            methylation_metadata_json["data"]["hits"]
        )

if metadata_records is None:
    raise ValueError(
        "Could not identify the file-record collection "
        "in the methylation metadata JSON."
    )

if not metadata_records:
    raise ValueError(
        "Methylation metadata JSON contains no file records."
    )

print(f"Metadata records: {len(metadata_records):,}")

first_metadata_record = metadata_records[0]

if not isinstance(first_metadata_record, dict):
    raise TypeError(
        "Metadata file records are not JSON objects."
    )

print("\nFirst metadata-record structure:")

for key, value in first_metadata_record.items():
    if isinstance(value, dict):
        print(
            f"- {key}: dict with keys "
            f"{list(value.keys())}"
        )

    elif isinstance(value, list):
        if value and isinstance(value[0], dict):
            print(
                f"- {key}: list[{len(value)}] "
                f"with first-item keys "
                f"{list(value[0].keys())}"
            )
        else:
            print(
                f"- {key}: list[{len(value)}]"
            )

    else:
        preview = repr(value)

        if len(preview) > 160:
            preview = preview[:157] + "..."

        print(
            f"- {key}: "
            f"{type(value).__name__} = {preview}"
        )

Metadata top-level type: list
Metadata records: 10,950

First metadata-record structure:
- data_format: str = 'TXT'
- access: str = 'open'
- associated_entities: list[1] with first-item keys ['entity_submitter_id', 'entity_type', 'case_id', 'entity_id']
- file_name: str = 'a3dad6d1-c405-4458-b5d2-63da84596781.methylation_array.sesame.level3betas.txt'
- submitter_id: str = '7890a8be-c106-474f-b708-a6cc5aa29a60'
- data_category: str = 'DNA Methylation'
- analysis: dict with keys ['workflow_version', 'updated_datetime', 'workflow_link', 'submitter_id', 'state', 'workflow_type', 'analysis_id', 'created_datetime']
- platform: str = 'Illumina Human Methylation 27'
- file_size: int = 763817
- md5sum: str = '8ca7304ae86d70ee30237fe0ffaee2d4'
- file_id: str = 'cf8a9d2e-0967-453d-9de9-8140b1d4afda'
- data_type: str = 'Methylation Beta Value'
- state: str = 'released'
- experimental_strategy: str = 'Methylation Array'


In [8]:
# =============================================================================
# Normalize and validate GDC methylation metadata
# =============================================================================

required_metadata_fields = {
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "state",
    "data_format",
    "access",
    "data_category",
    "data_type",
    "experimental_strategy",
    "platform",
    "analysis",
    "associated_entities",
}

records_with_missing_fields = []

for position, record in enumerate(metadata_records):
    missing_fields = required_metadata_fields - set(record)

    if missing_fields:
        records_with_missing_fields.append(
            {
                "record_position": position,
                "missing_fields": sorted(missing_fields),
            }
        )

if records_with_missing_fields:
    raise ValueError(
        "Metadata records with missing required fields were detected. "
        f"Affected records: {len(records_with_missing_fields):,}"
    )

metadata_file_records = []
associated_entity_records = []

for record in metadata_records:
    analysis = record["analysis"]
    associated_entities = record["associated_entities"]

    if not isinstance(analysis, dict):
        raise TypeError(
            f"Analysis metadata is not an object for file {record['file_id']}."
        )

    if not isinstance(associated_entities, list):
        raise TypeError(
            "Associated entities are not represented as a list for "
            f"file {record['file_id']}."
        )

    metadata_file_records.append(
        {
            "file_id": record["file_id"],
            "file_name": record["file_name"],
            "file_submitter_id": record.get("submitter_id"),
            "file_size_bytes": record["file_size"],
            "md5": record["md5sum"],
            "state": record["state"],
            "data_format": record["data_format"],
            "access": record["access"],
            "data_category": record["data_category"],
            "data_type": record["data_type"],
            "experimental_strategy": record["experimental_strategy"],
            "platform": record["platform"],
            "workflow_type": analysis.get("workflow_type"),
            "workflow_version": analysis.get("workflow_version"),
            "analysis_id": analysis.get("analysis_id"),
        }
    )

    for entity in associated_entities:
        associated_entity_records.append(
            {
                "file_id": record["file_id"],
                "entity_id": entity.get("entity_id"),
                "entity_submitter_id": entity.get(
                    "entity_submitter_id"
                ),
                "entity_type": entity.get("entity_type"),
                "case_id": entity.get("case_id"),
            }
        )

methylation_metadata_file_df = pd.DataFrame(
    metadata_file_records
)

methylation_associated_entity_df = pd.DataFrame(
    associated_entity_records
)

if methylation_metadata_file_df["file_id"].duplicated().any():
    raise ValueError(
        "Duplicated file IDs detected in normalized metadata."
    )

expected_metadata_values = {
    "data_category": "DNA Methylation",
    "data_type": "Methylation Beta Value",
    "experimental_strategy": "Methylation Array",
    "workflow_type": "SeSAMe Methylation Beta Estimation",
    "access": "open",
    "state": "released",
    "data_format": "TXT",
}

failed_metadata_checks = []

for column, expected_value in expected_metadata_values.items():
    observed_values = set(
        methylation_metadata_file_df[column]
        .astype("string")
        .fillna("<missing>")
    )

    passed = observed_values == {expected_value}

    print(
        f"{column}: {passed} "
        f"(observed={sorted(observed_values)})"
    )

    if not passed:
        failed_metadata_checks.append(column)

if failed_metadata_checks:
    raise ValueError(
        "Metadata query-definition checks failed: "
        + ", ".join(failed_metadata_checks)
    )

entities_per_file_summary = (
    methylation_associated_entity_df
    .groupby("file_id")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("associated_entities_per_file")
    .reset_index(name="file_count")
)

entity_type_summary = (
    methylation_associated_entity_df["entity_type"]
    .value_counts(dropna=False)
    .rename_axis("entity_type")
    .reset_index(name="association_count")
)

platform_file_summary = (
    methylation_metadata_file_df
    .groupby("platform", as_index=False)
    .agg(
        file_count=("file_id", "size"),
        total_size_bytes=("file_size_bytes", "sum"),
    )
    .sort_values("file_count", ascending=False)
)

platform_file_summary["total_size_gib"] = (
    platform_file_summary["total_size_bytes"] / 1024**3
)

print("\nMetadata normalized and query definition validated.")
print(
    f"File records:             "
    f"{len(methylation_metadata_file_df):,}"
)
print(
    f"Entity associations:      "
    f"{len(methylation_associated_entity_df):,}"
)

print("\nAssociated entities per file:")
display(entities_per_file_summary)

print("\nAssociated entity types:")
display(entity_type_summary)

print("\nInitial platform-level file coverage:")
display(platform_file_summary)

data_category: True (observed=['DNA Methylation'])
data_type: True (observed=['Methylation Beta Value'])
experimental_strategy: True (observed=['Methylation Array'])
workflow_type: True (observed=['SeSAMe Methylation Beta Estimation'])
access: True (observed=['open'])
state: True (observed=['released'])
data_format: True (observed=['TXT'])

Metadata normalized and query definition validated.
File records:             10,950
Entity associations:      10,950

Associated entities per file:


,associated_entities_per_file,file_count
0,1,10950



Associated entity types:


,entity_type,association_count
0,aliquot,10950



Initial platform-level file coverage:


,platform,file_count,total_size_bytes,total_size_gib
1,Illumina Human Methylation 450,8618,112853901884,105.103386
0,Illumina Human Methylation 27,2279,1752659375,1.632291
2,Illumina Methylation Epic v2,53,1351253876,1.258453


In [9]:
# =============================================================================
# Normalize and validate GDC methylation metadata
# =============================================================================

required_metadata_fields = {
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "state",
    "data_format",
    "access",
    "data_category",
    "data_type",
    "experimental_strategy",
    "platform",
    "analysis",
    "associated_entities",
}

records_with_missing_fields = []

for position, record in enumerate(metadata_records):
    missing_fields = required_metadata_fields - set(record)

    if missing_fields:
        records_with_missing_fields.append(
            {
                "record_position": position,
                "missing_fields": sorted(missing_fields),
            }
        )

if records_with_missing_fields:
    raise ValueError(
        "Metadata records with missing required fields were detected. "
        f"Affected records: {len(records_with_missing_fields):,}"
    )

metadata_file_records = []
associated_entity_records = []

for record in metadata_records:
    analysis = record["analysis"]
    associated_entities = record["associated_entities"]

    if not isinstance(analysis, dict):
        raise TypeError(
            f"Analysis metadata is not an object for file {record['file_id']}."
        )

    if not isinstance(associated_entities, list):
        raise TypeError(
            "Associated entities are not represented as a list for "
            f"file {record['file_id']}."
        )

    metadata_file_records.append(
        {
            "file_id": record["file_id"],
            "file_name": record["file_name"],
            "file_submitter_id": record.get("submitter_id"),
            "file_size_bytes": record["file_size"],
            "md5": record["md5sum"],
            "state": record["state"],
            "data_format": record["data_format"],
            "access": record["access"],
            "data_category": record["data_category"],
            "data_type": record["data_type"],
            "experimental_strategy": record["experimental_strategy"],
            "platform": record["platform"],
            "workflow_type": analysis.get("workflow_type"),
            "workflow_version": analysis.get("workflow_version"),
            "analysis_id": analysis.get("analysis_id"),
        }
    )

    for entity in associated_entities:
        associated_entity_records.append(
            {
                "file_id": record["file_id"],
                "entity_id": entity.get("entity_id"),
                "entity_submitter_id": entity.get(
                    "entity_submitter_id"
                ),
                "entity_type": entity.get("entity_type"),
                "case_id": entity.get("case_id"),
            }
        )

methylation_metadata_file_df = pd.DataFrame(
    metadata_file_records
)

methylation_associated_entity_df = pd.DataFrame(
    associated_entity_records
)

if methylation_metadata_file_df["file_id"].duplicated().any():
    raise ValueError(
        "Duplicated file IDs detected in normalized metadata."
    )

expected_metadata_values = {
    "data_category": "DNA Methylation",
    "data_type": "Methylation Beta Value",
    "experimental_strategy": "Methylation Array",
    "workflow_type": "SeSAMe Methylation Beta Estimation",
    "access": "open",
    "state": "released",
    "data_format": "TXT",
}

failed_metadata_checks = []

for column, expected_value in expected_metadata_values.items():
    observed_values = set(
        methylation_metadata_file_df[column]
        .astype("string")
        .fillna("<missing>")
    )

    passed = observed_values == {expected_value}

    print(
        f"{column}: {passed} "
        f"(observed={sorted(observed_values)})"
    )

    if not passed:
        failed_metadata_checks.append(column)

if failed_metadata_checks:
    raise ValueError(
        "Metadata query-definition checks failed: "
        + ", ".join(failed_metadata_checks)
    )

entities_per_file_summary = (
    methylation_associated_entity_df
    .groupby("file_id")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("associated_entities_per_file")
    .reset_index(name="file_count")
)

entity_type_summary = (
    methylation_associated_entity_df["entity_type"]
    .value_counts(dropna=False)
    .rename_axis("entity_type")
    .reset_index(name="association_count")
)

platform_file_summary = (
    methylation_metadata_file_df
    .groupby("platform", as_index=False)
    .agg(
        file_count=("file_id", "size"),
        total_size_bytes=("file_size_bytes", "sum"),
    )
    .sort_values("file_count", ascending=False)
)

platform_file_summary["total_size_gib"] = (
    platform_file_summary["total_size_bytes"] / 1024**3
)

print("\nMetadata normalized and query definition validated.")
print(
    f"File records:             "
    f"{len(methylation_metadata_file_df):,}"
)
print(
    f"Entity associations:      "
    f"{len(methylation_associated_entity_df):,}"
)

print("\nAssociated entities per file:")
print(entities_per_file_summary)

print("\nAssociated entity types:")
print(entity_type_summary)

print("\nInitial platform-level file coverage:")
print(platform_file_summary)

data_category: True (observed=['DNA Methylation'])
data_type: True (observed=['Methylation Beta Value'])
experimental_strategy: True (observed=['Methylation Array'])
workflow_type: True (observed=['SeSAMe Methylation Beta Estimation'])
access: True (observed=['open'])
state: True (observed=['released'])
data_format: True (observed=['TXT'])

Metadata normalized and query definition validated.
File records:             10,950
Entity associations:      10,950

Associated entities per file:
   associated_entities_per_file  file_count
0                             1       10950

Associated entity types:
  entity_type  association_count
0     aliquot              10950

Initial platform-level file coverage:
                         platform  file_count  total_size_bytes  \
1  Illumina Human Methylation 450        8618      112853901884   
0   Illumina Human Methylation 27        2279        1752659375   
2    Illumina Methylation Epic v2          53        1351253876   

   total_size_gib  


In [10]:
# =============================================================================
# Validate cross-source consistency and build methylation file index
# =============================================================================

manifest_index_df = methylation_manifest_df.rename(
    columns={
        "id": "file_id",
        "filename": "file_name",
        "md5": "md5",
        "size": "file_size_bytes",
        "state": "state",
    }
)[
    [
        "file_id",
        "file_name",
        "md5",
        "file_size_bytes",
        "state",
    ]
].copy()

sample_sheet_index_df = methylation_sample_sheet_df.rename(
    columns={
        "File ID": "file_id",
        "File Name": "file_name",
        "Data Category": "data_category",
        "Data Type": "data_type",
        "Project ID": "project_id",
        "Case ID": "case_submitter_id",
        "Sample ID": "sample_submitter_id",
        "Tissue Type": "tissue_type",
        "Tumor Descriptor": "tumor_descriptor",
        "Specimen Type": "specimen_type",
        "Preservation Method": "preservation_method",
    }
).copy()

manifest_ids = set(manifest_index_df["file_id"])
sample_sheet_ids = set(sample_sheet_index_df["file_id"])
metadata_ids = set(methylation_metadata_file_df["file_id"])
associated_entity_ids = set(
    methylation_associated_entity_df["file_id"]
)

id_set_comparison = {
    "manifest_only_vs_sample_sheet": len(
        manifest_ids - sample_sheet_ids
    ),
    "sample_sheet_only_vs_manifest": len(
        sample_sheet_ids - manifest_ids
    ),
    "manifest_only_vs_metadata": len(
        manifest_ids - metadata_ids
    ),
    "metadata_only_vs_manifest": len(
        metadata_ids - manifest_ids
    ),
    "manifest_without_associated_entity": len(
        manifest_ids - associated_entity_ids
    ),
    "associated_entity_without_manifest": len(
        associated_entity_ids - manifest_ids
    ),
}

print("File-ID set comparison:")

for comparison_name, count in id_set_comparison.items():
    print(f"- {comparison_name}: {count:,}")

if any(id_set_comparison.values()):
    raise ValueError(
        "Manifest, sample sheet, metadata, and associated entities "
        "do not contain identical file-ID sets."
    )

manifest_metadata_check_df = manifest_index_df.merge(
    methylation_metadata_file_df[
        [
            "file_id",
            "file_name",
            "md5",
            "file_size_bytes",
            "state",
        ]
    ],
    on="file_id",
    how="inner",
    suffixes=("_manifest", "_metadata"),
    validate="one_to_one",
)

manifest_metadata_mismatches = {
    "file_name": (
        manifest_metadata_check_df["file_name_manifest"]
        != manifest_metadata_check_df["file_name_metadata"]
    ).sum(),
    "md5": (
        manifest_metadata_check_df["md5_manifest"]
        != manifest_metadata_check_df["md5_metadata"]
    ).sum(),
    "file_size_bytes": (
        manifest_metadata_check_df["file_size_bytes_manifest"]
        != manifest_metadata_check_df["file_size_bytes_metadata"]
    ).sum(),
    "state": (
        manifest_metadata_check_df["state_manifest"]
        != manifest_metadata_check_df["state_metadata"]
    ).sum(),
}

sample_metadata_check_df = sample_sheet_index_df.merge(
    methylation_metadata_file_df[
        [
            "file_id",
            "file_name",
            "data_category",
            "data_type",
        ]
    ],
    on="file_id",
    how="inner",
    suffixes=("_sample_sheet", "_metadata"),
    validate="one_to_one",
)

sample_metadata_mismatches = {
    "file_name": (
        sample_metadata_check_df["file_name_sample_sheet"]
        != sample_metadata_check_df["file_name_metadata"]
    ).sum(),
    "data_category": (
        sample_metadata_check_df["data_category_sample_sheet"]
        != sample_metadata_check_df["data_category_metadata"]
    ).sum(),
    "data_type": (
        sample_metadata_check_df["data_type_sample_sheet"]
        != sample_metadata_check_df["data_type_metadata"]
    ).sum(),
}

print("\nManifest-versus-metadata mismatches:")

for field, count in manifest_metadata_mismatches.items():
    print(f"- {field}: {count:,}")

print("\nSample-sheet-versus-metadata mismatches:")

for field, count in sample_metadata_mismatches.items():
    print(f"- {field}: {count:,}")

if any(manifest_metadata_mismatches.values()):
    raise ValueError(
        "Manifest and metadata contain inconsistent file attributes."
    )

if any(sample_metadata_mismatches.values()):
    raise ValueError(
        "Sample sheet and metadata contain inconsistent file attributes."
    )

if methylation_associated_entity_df["file_id"].duplicated().any():
    raise ValueError(
        "More than one associated entity was detected for a file."
    )

if not (
    methylation_associated_entity_df["entity_type"] == "aliquot"
).all():
    raise ValueError(
        "Not all methylation files are associated with aliquots."
    )

aliquot_index_df = methylation_associated_entity_df.rename(
    columns={
        "entity_id": "aliquot_uuid",
        "entity_submitter_id": "aliquot_submitter_id",
        "case_id": "case_uuid",
    }
)[
    [
        "file_id",
        "case_uuid",
        "aliquot_uuid",
        "aliquot_submitter_id",
    ]
].copy()

methylation_file_index_df = (
    manifest_index_df
    .merge(
        sample_sheet_index_df,
        on="file_id",
        how="inner",
        suffixes=("", "_sample_sheet"),
        validate="one_to_one",
    )
)

if not (
    methylation_file_index_df["file_name"]
    == methylation_file_index_df["file_name_sample_sheet"]
).all():
    raise ValueError(
        "Filename mismatch while building methylation file index."
    )

methylation_file_index_df = (
    methylation_file_index_df
    .drop(columns="file_name_sample_sheet")
    .merge(
        methylation_metadata_file_df[
            [
                "file_id",
                "file_submitter_id",
                "data_format",
                "access",
                "experimental_strategy",
                "platform",
                "workflow_type",
                "workflow_version",
                "analysis_id",
            ]
        ],
        on="file_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        aliquot_index_df,
        on="file_id",
        how="inner",
        validate="one_to_one",
    )
)

methylation_file_index_df = methylation_file_index_df[
    [
        "file_id",
        "file_name",
        "file_submitter_id",
        "md5",
        "file_size_bytes",
        "state",
        "data_format",
        "access",
        "data_category",
        "data_type",
        "experimental_strategy",
        "workflow_type",
        "workflow_version",
        "analysis_id",
        "platform",
        "project_id",
        "case_uuid",
        "case_submitter_id",
        "sample_submitter_id",
        "aliquot_uuid",
        "aliquot_submitter_id",
        "tissue_type",
        "tumor_descriptor",
        "specimen_type",
        "preservation_method",
    ]
].sort_values(
    [
        "project_id",
        "case_submitter_id",
        "sample_submitter_id",
        "platform",
        "file_id",
    ]
).reset_index(drop=True)

print("\nCanonical methylation file index built.")
print(f"Rows:             {len(methylation_file_index_df):,}")
print(
    f"Projects:         "
    f"{methylation_file_index_df['project_id'].nunique():,}"
)
print(
    f"Cases:            "
    f"{methylation_file_index_df['case_submitter_id'].nunique():,}"
)
print(
    f"Samples:          "
    f"{methylation_file_index_df['sample_submitter_id'].nunique():,}"
)
print(
    f"Aliquots:         "
    f"{methylation_file_index_df['aliquot_uuid'].nunique():,}"
)
print(
    f"Platforms:        "
    f"{methylation_file_index_df['platform'].nunique():,}"
)

display(methylation_file_index_df.head())

File-ID set comparison:
- manifest_only_vs_sample_sheet: 0
- sample_sheet_only_vs_manifest: 0
- manifest_only_vs_metadata: 0
- metadata_only_vs_manifest: 0
- manifest_without_associated_entity: 0
- associated_entity_without_manifest: 0

Manifest-versus-metadata mismatches:
- file_name: 0
- md5: 0
- file_size_bytes: 0
- state: 0

Sample-sheet-versus-metadata mismatches:
- file_name: 0
- data_category: 0
- data_type: 0

Canonical methylation file index built.
Rows:             10,950
Projects:         33
Cases:            10,610
Samples:          10,656
Aliquots:         10,756
Platforms:        3


,file_id,file_name,file_submitter_id,md5,file_size_bytes,state,data_format,access,data_category,data_type,...,project_id,case_uuid,case_submitter_id,sample_submitter_id,aliquot_uuid,aliquot_submitter_id,tissue_type,tumor_descriptor,specimen_type,preservation_method
0,060eea58-77fe-4661-81f2-484f145661ac,bb52bb9a-0ff9-4ab3-a91a-dd6082a70905.methylati...,fdd2c889-6f27-4c7f-ba37-7ee7f95e4fb2,e9fe0b08bcd1c138c19cd0d5b02a00ea,13127970,released,TXT,open,DNA Methylation,Methylation Beta Value,...,TCGA-ACC,b3164f7b-c826-4e08-9ee6-8ff96d29b913,TCGA-OR-A5J1,TCGA-OR-A5J1-01A,8690b8a2-d020-4172-a2a8-1f34bf6cb89c,TCGA-OR-A5J1-01A-11D-A29J-05,Tumor,Primary,Unknown,OCT
1,0a887275-50ba-4162-9422-14a3a614d18c,571c403a-8b29-4b29-b74b-eddbc96c90d3.methylati...,47283d3e-b2f7-4195-bbec-642587d75f47,5960d9ed32857ad3e3d0ee47ad595ceb,12958699,released,TXT,open,DNA Methylation,Methylation Beta Value,...,TCGA-ACC,8e7c2e31-d085-4b75-a970-162526dd07a0,TCGA-OR-A5J2,TCGA-OR-A5J2-01A,4a320e08-0101-4ddc-85c6-cbb9e59fb838,TCGA-OR-A5J2-01A-11D-A29J-05,Tumor,Primary,Unknown,OCT
2,5f15dfb7-a6b1-49ee-b90b-225866d84de6,74a6331a-773e-4d74-86c6-04a078a78403.methylati...,ba9e87ef-f3eb-4d47-8126-dc9ad5818b29,5da1cfab12166ad7413b907eaa4872a3,13102492,released,TXT,open,DNA Methylation,Methylation Beta Value,...,TCGA-ACC,dfd687bc-6e69-42f7-af94-d17fc150d1a1,TCGA-OR-A5J3,TCGA-OR-A5J3-01A,8be135eb-a9cd-4f7e-bfa1-a7fae79d57b7,TCGA-OR-A5J3-01A-11D-A29J-05,Tumor,Primary,Unknown,OCT
3,d2dcc333-1e90-4f63-b9c1-4f970be8056b,05d38477-f2dd-4832-894c-8b0fc8fbcb62.methylati...,933508c6-7b83-429e-981e-04f616fe0a74,dab486c4435c2df837abd8f6ef9c9130,13132453,released,TXT,open,DNA Methylation,Methylation Beta Value,...,TCGA-ACC,5f3e2974-f1df-47a2-8a8a-29bb525eeef6,TCGA-OR-A5J4,TCGA-OR-A5J4-01A,6abdb597-d4a4-42e9-8e33-091e2b4fa901,TCGA-OR-A5J4-01A-11D-A29J-05,Tumor,Primary,Unknown,OCT
4,548e765a-f532-4316-b87c-93dee6bc2293,2a0baa86-d3d0-4be1-ab6a-bf7a9d820bf4.methylati...,33d04e8d-6310-42fa-93c4-38608d273f7c,2cfbe938858f504cac387494550d5068,13172228,released,TXT,open,DNA Methylation,Methylation Beta Value,...,TCGA-ACC,802dbd0d-ef07-4c91-ab8d-1dd39532e947,TCGA-OR-A5J5,TCGA-OR-A5J5-01A,1269b1c0-3b7a-4c18-b180-9e644e3c37a7,TCGA-OR-A5J5-01A-11D-A29J-05,Tumor,Primary,Unknown,OCT


In [12]:
# =============================================================================
# Assess global platform coverage and identifier multiplicity
# =============================================================================

total_methylation_cases = (
    methylation_file_index_df["case_submitter_id"].nunique()
)

platform_coverage_df = (
    methylation_file_index_df
    .groupby("platform", as_index=False)
    .agg(
        n_files=("file_id", "nunique"),
        n_aliquots=("aliquot_uuid", "nunique"),
        n_samples=("sample_submitter_id", "nunique"),
        n_cases=("case_submitter_id", "nunique"),
        n_projects=("project_id", "nunique"),
        total_size_bytes=("file_size_bytes", "sum"),
        median_file_size_bytes=("file_size_bytes", "median"),
    )
)

platform_coverage_df["case_coverage_pct"] = (
    100
    * platform_coverage_df["n_cases"]
    / total_methylation_cases
)

platform_coverage_df["total_size_gib"] = (
    platform_coverage_df["total_size_bytes"] / 1024**3
)

platform_coverage_df["median_file_size_mib"] = (
    platform_coverage_df["median_file_size_bytes"] / 1024**2
)

platform_coverage_df = (
    platform_coverage_df[
        [
            "platform",
            "n_files",
            "n_aliquots",
            "n_samples",
            "n_cases",
            "n_projects",
            "case_coverage_pct",
            "total_size_gib",
            "median_file_size_mib",
        ]
    ]
    .sort_values(
        ["n_cases", "n_files"],
        ascending=False,
    )
    .reset_index(drop=True)
)

files_per_aliquot_summary = (
    methylation_file_index_df
    .groupby("aliquot_uuid")["file_id"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("files_per_aliquot")
    .reset_index(name="aliquot_count")
)

files_per_sample_summary = (
    methylation_file_index_df
    .groupby("sample_submitter_id")["file_id"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("files_per_sample")
    .reset_index(name="sample_count")
)

files_per_case_summary = (
    methylation_file_index_df
    .groupby("case_submitter_id")["file_id"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("files_per_case")
    .reset_index(name="case_count")
)

platforms_per_case_summary = (
    methylation_file_index_df
    .groupby("case_submitter_id")["platform"]
    .nunique()
    .value_counts()
    .sort_index()
    .rename_axis("platforms_per_case")
    .reset_index(name="case_count")
)

print("Global platform coverage:")
display(platform_coverage_df)

print("\nFiles per aliquot:")
display(files_per_aliquot_summary)

print("\nFiles per sample:")
display(files_per_sample_summary)

print("\nFiles per case:")
display(files_per_case_summary)

print("\nPlatforms represented per case:")
display(platforms_per_case_summary)

Global platform coverage:


,platform,n_files,n_aliquots,n_samples,n_cases,n_projects,case_coverage_pct,total_size_gib,median_file_size_mib
0,Illumina Human Methylation 450,8618,8618,8598,8555,33,80.631480,105.103386,12.530412
1,Illumina Human Methylation 27,2279,2279,2271,2270,12,21.394910,1.632291,0.735043
2,Illumina Methylation Epic v2,53,53,53,53,1,0.499529,1.258453,24.345627



Files per aliquot:


,files_per_aliquot,aliquot_count
0,1,10562
1,2,194



Files per sample:


,files_per_sample,sample_count
0,1,10367
1,2,287
2,3,1
3,6,1



Files per case:


,files_per_case,case_count
0,1,10307
1,2,270
2,3,31
3,4,1
4,6,1



Platforms represented per case:


,platforms_per_case,case_count
0,1,10342
1,2,268


In [14]:
# =============================================================================
# Quantify cross-platform case overlap and exclusive coverage
# =============================================================================

platform_label_map = {
    "Illumina Human Methylation 27": "HM27",
    "Illumina Human Methylation 450": "HM450",
    "Illumina Methylation Epic v2": "EPICv2",
}

case_platform_pairs_df = (
    methylation_file_index_df[
        [
            "case_submitter_id",
            "project_id",
            "platform",
        ]
    ]
    .drop_duplicates()
    .copy()
)

case_platform_pairs_df["platform_label"] = (
    case_platform_pairs_df["platform"].map(
        platform_label_map
    )
)

if case_platform_pairs_df["platform_label"].isna().any():
    unknown_platforms = sorted(
        case_platform_pairs_df.loc[
            case_platform_pairs_df["platform_label"].isna(),
            "platform",
        ].unique()
    )

    raise ValueError(
        "Unmapped methylation platforms detected: "
        + ", ".join(unknown_platforms)
    )

projects_per_case = (
    case_platform_pairs_df
    .groupby("case_submitter_id")["project_id"]
    .nunique()
)

if (projects_per_case != 1).any():
    raise ValueError(
        "At least one TCGA case is associated with multiple projects."
    )

case_project_df = (
    case_platform_pairs_df[
        ["case_submitter_id", "project_id"]
    ]
    .drop_duplicates()
)

platform_order = [
    "HM27",
    "HM450",
    "EPICv2",
]

case_platform_presence_df = (
    pd.crosstab(
        index=case_platform_pairs_df["case_submitter_id"],
        columns=case_platform_pairs_df["platform_label"],
    )
    .reindex(
        columns=platform_order,
        fill_value=0,
    )
    .gt(0)
)

case_platform_membership_df = (
    case_platform_presence_df
    .reset_index()
    .merge(
        case_project_df,
        on="case_submitter_id",
        how="left",
        validate="one_to_one",
    )
)

case_platform_membership_df["platform_combination"] = (
    case_platform_membership_df[platform_order]
    .apply(
        lambda row: " + ".join(
            platform
            for platform in platform_order
            if row[platform]
        ),
        axis=1,
    )
)

platform_combination_summary_df = (
    case_platform_membership_df
    .groupby(
        "platform_combination",
        as_index=False,
    )
    .agg(
        n_cases=("case_submitter_id", "nunique"),
        n_projects=("project_id", "nunique"),
    )
    .sort_values(
        "n_cases",
        ascending=False,
    )
    .reset_index(drop=True)
)

platform_presence_integer_df = (
    case_platform_presence_df
    .astype("int64")
)

pairwise_case_overlap_df = (
    platform_presence_integer_df.T
    @ platform_presence_integer_df
)

pairwise_case_overlap_df.index.name = "platform"
pairwise_case_overlap_df.columns.name = "platform"

print("Case-level platform combinations:")
display(platform_combination_summary_df)

print("\nPairwise case-overlap matrix:")
display(pairwise_case_overlap_df)

print(
    "\nTotal unique methylation cases:",
    f"{len(case_platform_membership_df):,}",
)

Case-level platform combinations:


,platform_combination,n_cases,n_projects
0,HM450,8289,32
1,HM27,2053,11
2,HM27 + HM450,215,8
3,HM450 + EPICv2,51,1
4,HM27 + EPICv2,2,1



Pairwise case-overlap matrix:


platform,HM27,HM450,EPICv2
platform,,,
HM27,2270,215,2
HM450,215,8555,51
EPICv2,2,51,53



Total unique methylation cases: 10,610


In [16]:
# =============================================================================
# Assess project-level platform coverage and HM450 retention
# =============================================================================

project_case_membership_df = (
    case_platform_membership_df.copy()
)

project_case_membership_df["hm27_only"] = (
    project_case_membership_df["platform_combination"]
    == "HM27"
)

project_case_membership_df["hm450_only"] = (
    project_case_membership_df["platform_combination"]
    == "HM450"
)

project_case_membership_df["hm27_hm450"] = (
    project_case_membership_df["platform_combination"]
    == "HM27 + HM450"
)

project_case_membership_df["hm450_epicv2"] = (
    project_case_membership_df["platform_combination"]
    == "HM450 + EPICv2"
)

project_case_membership_df["hm27_epicv2"] = (
    project_case_membership_df["platform_combination"]
    == "HM27 + EPICv2"
)

project_platform_coverage_df = (
    project_case_membership_df
    .groupby("project_id", as_index=False)
    .agg(
        total_methylation_cases=(
            "case_submitter_id",
            "nunique",
        ),
        hm27_cases=("HM27", "sum"),
        hm450_cases=("HM450", "sum"),
        epicv2_cases=("EPICv2", "sum"),
        hm27_only_cases=("hm27_only", "sum"),
        hm450_only_cases=("hm450_only", "sum"),
        hm27_hm450_cases=("hm27_hm450", "sum"),
        hm450_epicv2_cases=("hm450_epicv2", "sum"),
        hm27_epicv2_cases=("hm27_epicv2", "sum"),
    )
)

project_platform_coverage_df["cases_without_hm450"] = (
    project_platform_coverage_df["total_methylation_cases"]
    - project_platform_coverage_df["hm450_cases"]
)

project_platform_coverage_df["hm450_retention_pct"] = (
    100
    * project_platform_coverage_df["hm450_cases"]
    / project_platform_coverage_df["total_methylation_cases"]
)

project_platform_coverage_df["hm27_incremental_pct"] = (
    100
    * project_platform_coverage_df["cases_without_hm450"]
    / project_platform_coverage_df["total_methylation_cases"]
)

project_platform_coverage_df = (
    project_platform_coverage_df
    .sort_values(
        [
            "hm450_retention_pct",
            "total_methylation_cases",
        ],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

projects_without_hm450 = project_platform_coverage_df.loc[
    project_platform_coverage_df["hm450_cases"] == 0,
    "project_id",
].tolist()

print("Project-level platform coverage:")
print(
    f"Projects assessed:        "
    f"{len(project_platform_coverage_df):,}"
)
print(
    f"Projects without HM450:   "
    f"{len(projects_without_hm450):,}"
)

if projects_without_hm450:
    print(
        "Projects lacking HM450:  "
        + ", ".join(projects_without_hm450)
    )

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
):
    display(project_platform_coverage_df)

Project-level platform coverage:
Projects assessed:        33
Projects without HM450:   0


,project_id,total_methylation_cases,hm27_cases,hm450_cases,epicv2_cases,hm27_only_cases,hm450_only_cases,hm27_hm450_cases,hm450_epicv2_cases,hm27_epicv2_cases,cases_without_hm450,hm450_retention_pct,hm27_incremental_pct
0,TCGA-OV,592,582,10,0,582,10,0,0,0,582,1.689189,98.310811
1,TCGA-GBM,422,287,140,0,282,135,5,0,0,282,33.175355,66.824645
2,TCGA-READ,165,68,98,0,67,97,1,0,0,67,59.393939,40.606061
3,TCGA-KIRC,535,219,319,0,216,316,3,0,0,216,59.626168,40.373832
4,TCGA-COAD,457,166,295,0,162,291,4,0,0,162,64.551422,35.448578
5,TCGA-BRCA,1097,314,784,0,313,783,1,0,0,313,71.467639,28.532361
6,TCGA-LUSC,503,133,370,0,133,370,0,0,0,133,73.558648,26.441352
7,TCGA-UCEC,547,117,431,0,116,430,1,0,0,116,78.793419,21.206581
8,TCGA-LUAD,578,126,458,53,118,401,6,51,2,120,79.238754,20.761246
9,TCGA-STAD,443,48,395,0,48,395,0,0,0,48,89.164786,10.835214


In [18]:
# =============================================================================
# Assess global RNA-seq overlap by methylation-platform strategy
# =============================================================================

required_rna_index_columns = {
    "file_id",
    "project_id",
    "case_id",
    "sample_id",
}

missing_rna_index_columns = (
    required_rna_index_columns
    - set(rna_file_index_df.columns)
)

if missing_rna_index_columns:
    raise ValueError(
        "RNA-seq file index is missing required columns: "
        + ", ".join(sorted(missing_rna_index_columns))
    )

rna_required_nulls = (
    rna_file_index_df[
        [
            "file_id",
            "project_id",
            "case_id",
            "sample_id",
        ]
    ]
    .isna()
    .sum()
)

if (rna_required_nulls > 0).any():
    raise ValueError(
        "RNA-seq file index contains missing required identifiers."
    )

rna_case_project_counts = (
    rna_file_index_df
    .groupby("case_id")["project_id"]
    .nunique()
)

if (rna_case_project_counts != 1).any():
    raise ValueError(
        "At least one RNA-seq case is associated with multiple projects."
    )

rna_case_df = (
    rna_file_index_df[
        [
            "case_id",
            "project_id",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "case_id": "case_submitter_id",
        }
    )
)

methylation_case_df = (
    case_platform_membership_df[
        [
            "case_submitter_id",
            "project_id",
            "HM27",
            "HM450",
            "EPICv2",
            "platform_combination",
        ]
    ]
    .copy()
)

overlapping_case_project_check_df = (
    rna_case_df.merge(
        methylation_case_df[
            [
                "case_submitter_id",
                "project_id",
            ]
        ],
        on="case_submitter_id",
        how="inner",
        suffixes=("_rna", "_methylation"),
        validate="one_to_one",
    )
)

project_mismatches = (
    overlapping_case_project_check_df["project_id_rna"]
    != overlapping_case_project_check_df[
        "project_id_methylation"
    ]
)

if project_mismatches.any():
    raise ValueError(
        "RNA-seq and methylation assign overlapping cases "
        "to different TCGA projects."
    )

rna_case_ids = set(rna_case_df["case_submitter_id"])
methylation_case_ids = set(
    methylation_case_df["case_submitter_id"]
)

global_overlap_case_ids = (
    rna_case_ids & methylation_case_ids
)

strategy_masks = {
    "Any methylation platform": pd.Series(
        True,
        index=methylation_case_df.index,
    ),
    "HM450": methylation_case_df["HM450"],
    "HM27": methylation_case_df["HM27"],
    "EPICv2": methylation_case_df["EPICv2"],
    "HM27 or HM450": (
        methylation_case_df["HM27"]
        | methylation_case_df["HM450"]
    ),
}

strategy_overlap_records = []

for strategy_name, strategy_mask in strategy_masks.items():
    strategy_case_ids = set(
        methylation_case_df.loc[
            strategy_mask,
            "case_submitter_id",
        ]
    )

    overlapping_rna_case_ids = (
        strategy_case_ids & rna_case_ids
    )

    strategy_overlap_records.append(
        {
            "strategy": strategy_name,
            "n_methylation_cases": len(
                strategy_case_ids
            ),
            "n_rna_overlap_cases": len(
                overlapping_rna_case_ids
            ),
            "rna_case_coverage_pct": (
                100
                * len(overlapping_rna_case_ids)
                / len(rna_case_ids)
            ),
            "methylation_cases_with_rna_pct": (
                100
                * len(overlapping_rna_case_ids)
                / len(strategy_case_ids)
            ),
        }
    )

strategy_rna_overlap_df = pd.DataFrame(
    strategy_overlap_records
).sort_values(
    "n_rna_overlap_cases",
    ascending=False,
).reset_index(drop=True)

print("Cross-modality case overlap:")
print(f"RNA-seq cases:             {len(rna_case_ids):,}")
print(
    f"Any methylation cases:     "
    f"{len(methylation_case_ids):,}"
)
print(
    f"RNA–methylation overlap:   "
    f"{len(global_overlap_case_ids):,}"
)
print(
    f"RNA-only cases:            "
    f"{len(rna_case_ids - methylation_case_ids):,}"
)
print(
    f"Methylation-only cases:    "
    f"{len(methylation_case_ids - rna_case_ids):,}"
)

print("\nRNA-seq overlap by methylation-platform strategy:")
display(strategy_rna_overlap_df)

Cross-modality case overlap:
RNA-seq cases:             10,122
Any methylation cases:     10,610
RNA–methylation overlap:   10,054
RNA-only cases:            68
Methylation-only cases:    556

RNA-seq overlap by methylation-platform strategy:


,strategy,n_methylation_cases,n_rna_overlap_cases,rna_case_coverage_pct,methylation_cases_with_rna_pct
0,Any methylation platform,10610,10054,99.328196,94.759661
1,HM27 or HM450,10610,10054,99.328196,94.759661
2,HM450,8555,8376,82.750445,97.907656
3,HM27,2270,1844,18.217744,81.233480
4,EPICv2,53,52,0.513732,98.113208


In [22]:
# =============================================================================
# Assess project-level RNA-seq overlap by methylation platform
# =============================================================================

rna_case_platform_df = (
    rna_case_df
    .merge(
        methylation_case_df[
            [
                "case_submitter_id",
                "HM27",
                "HM450",
                "EPICv2",
                "platform_combination",
            ]
        ],
        on="case_submitter_id",
        how="left",
        validate="one_to_one",
    )
)

for platform_column in ["HM27", "HM450", "EPICv2"]:
    rna_case_platform_df[platform_column] = (
        rna_case_platform_df[platform_column].eq(True)
    )


rna_case_platform_df["has_any_methylation"] = (
    rna_case_platform_df["platform_combination"].notna()
)

project_rna_methylation_coverage_df = (
    rna_case_platform_df
    .groupby("project_id", as_index=False)
    .agg(
        n_rna_cases=("case_submitter_id", "nunique"),
        n_any_methylation_cases=(
            "has_any_methylation",
            "sum",
        ),
        n_hm27_cases=("HM27", "sum"),
        n_hm450_cases=("HM450", "sum"),
        n_epicv2_cases=("EPICv2", "sum"),
    )
)

project_rna_methylation_coverage_df[
    "n_rna_without_methylation"
] = (
    project_rna_methylation_coverage_df["n_rna_cases"]
    - project_rna_methylation_coverage_df[
        "n_any_methylation_cases"
    ]
)

project_rna_methylation_coverage_df[
    "n_non_hm450_incremental_cases"
] = (
    project_rna_methylation_coverage_df[
        "n_any_methylation_cases"
    ]
    - project_rna_methylation_coverage_df[
        "n_hm450_cases"
    ]
)

project_rna_methylation_coverage_df[
    "any_methylation_coverage_pct"
] = (
    100
    * project_rna_methylation_coverage_df[
        "n_any_methylation_cases"
    ]
    / project_rna_methylation_coverage_df["n_rna_cases"]
)

project_rna_methylation_coverage_df[
    "hm450_rna_coverage_pct"
] = (
    100
    * project_rna_methylation_coverage_df[
        "n_hm450_cases"
    ]
    / project_rna_methylation_coverage_df["n_rna_cases"]
)

project_rna_methylation_coverage_df[
    "hm27_rna_coverage_pct"
] = (
    100
    * project_rna_methylation_coverage_df[
        "n_hm27_cases"
    ]
    / project_rna_methylation_coverage_df["n_rna_cases"]
)

project_rna_methylation_coverage_df[
    "hm450_retention_within_multiomic_pct"
] = (
    100
    * project_rna_methylation_coverage_df[
        "n_hm450_cases"
    ]
    / project_rna_methylation_coverage_df[
        "n_any_methylation_cases"
    ]
)

project_rna_methylation_coverage_df = (
    project_rna_methylation_coverage_df
    .sort_values(
        [
            "hm450_rna_coverage_pct",
            "n_rna_cases",
        ],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print("Project-level RNA–methylation coverage:")
print(
    "RNA cases with any methylation: "
    f"{project_rna_methylation_coverage_df['n_any_methylation_cases'].sum():,}"
)
print(
    "RNA cases with HM450:           "
    f"{project_rna_methylation_coverage_df['n_hm450_cases'].sum():,}"
)
print(
    "Incremental non-HM450 cases:    "
    f"{project_rna_methylation_coverage_df['n_non_hm450_incremental_cases'].sum():,}"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
):
    display(project_rna_methylation_coverage_df)

Project-level RNA–methylation coverage:
RNA cases with any methylation: 10,054
RNA cases with HM450:           8,376
Incremental non-HM450 cases:    1,678


,project_id,n_rna_cases,n_any_methylation_cases,n_hm27_cases,n_hm450_cases,n_epicv2_cases,n_rna_without_methylation,n_non_hm450_incremental_cases,any_methylation_coverage_pct,hm450_rna_coverage_pct,hm27_rna_coverage_pct,hm450_retention_within_multiomic_pct
0,TCGA-OV,426,426,417,9,0,0,417,100.000000,2.112676,97.887324,2.112676
1,TCGA-GBM,284,230,152,80,0,54,150,80.985915,28.169014,53.521127,34.782609
2,TCGA-READ,166,164,67,98,0,2,66,98.795181,59.036145,40.361446,59.756098
3,TCGA-KIRC,533,532,217,318,0,1,214,99.812383,59.662289,40.712946,59.774436
4,TCGA-COAD,458,455,164,295,0,3,160,99.344978,64.410480,35.807860,64.835165
5,TCGA-BRCA,1095,1094,313,782,0,1,312,99.908676,71.415525,28.584475,71.480804
6,TCGA-LUSC,501,500,130,370,0,1,130,99.800399,73.852295,25.948104,74.000000
7,TCGA-UCEC,545,545,116,430,0,0,115,100.000000,78.899083,21.284404,78.899083
8,TCGA-LUAD,517,514,65,455,52,3,59,99.419729,88.007737,12.572534,88.521401
9,TCGA-STAD,412,412,39,373,0,0,39,100.000000,90.533981,9.466019,90.533981


In [23]:
# =============================================================================
# Assess cross-platform pairing at case, sample, and aliquot levels
# =============================================================================

def build_platform_presence(
    data: pd.DataFrame,
    identifier_column: str,
) -> pd.DataFrame:
    """Build a Boolean platform-presence matrix for one identifier level."""
    identifier_platform_df = (
        data[
            [
                identifier_column,
                "platform",
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    identifier_platform_df["platform_label"] = (
        identifier_platform_df["platform"].map(
            platform_label_map
        )
    )

    if identifier_platform_df["platform_label"].isna().any():
        raise ValueError(
            f"Unmapped platform detected at {identifier_column} level."
        )

    return (
        pd.crosstab(
            index=identifier_platform_df[identifier_column],
            columns=identifier_platform_df["platform_label"],
        )
        .reindex(
            columns=platform_order,
            fill_value=0,
        )
        .gt(0)
    )


case_platform_presence_detailed_df = build_platform_presence(
    methylation_file_index_df,
    "case_submitter_id",
)

sample_platform_presence_df = build_platform_presence(
    methylation_file_index_df,
    "sample_submitter_id",
)

aliquot_platform_presence_df = build_platform_presence(
    methylation_file_index_df,
    "aliquot_uuid",
)

platform_pairs = [
    ("HM27", "HM450"),
    ("HM27", "EPICv2"),
    ("HM450", "EPICv2"),
]

pairing_records = []

for platform_a, platform_b in platform_pairs:
    pairing_records.append(
        {
            "platform_pair": (
                f"{platform_a} + {platform_b}"
            ),
            "paired_cases": int(
                (
                    case_platform_presence_detailed_df[platform_a]
                    & case_platform_presence_detailed_df[platform_b]
                ).sum()
            ),
            "paired_samples": int(
                (
                    sample_platform_presence_df[platform_a]
                    & sample_platform_presence_df[platform_b]
                ).sum()
            ),
            "paired_aliquots": int(
                (
                    aliquot_platform_presence_df[platform_a]
                    & aliquot_platform_presence_df[platform_b]
                ).sum()
            ),
        }
    )

cross_platform_pairing_df = pd.DataFrame(
    pairing_records
)

hm27_hm450_case_ids = set(
    case_platform_presence_detailed_df.index[
        case_platform_presence_detailed_df["HM27"]
        & case_platform_presence_detailed_df["HM450"]
    ]
)

hm27_hm450_sample_ids = set(
    sample_platform_presence_df.index[
        sample_platform_presence_df["HM27"]
        & sample_platform_presence_df["HM450"]
    ]
)

hm27_hm450_aliquot_ids = set(
    aliquot_platform_presence_df.index[
        aliquot_platform_presence_df["HM27"]
        & aliquot_platform_presence_df["HM450"]
    ]
)

paired_cases_by_project_df = (
    methylation_file_index_df.loc[
        methylation_file_index_df["case_submitter_id"].isin(
            hm27_hm450_case_ids
        )
    ]
    .groupby("project_id")["case_submitter_id"]
    .nunique()
    .rename("paired_cases")
    .reset_index()
)

paired_samples_by_project_df = (
    methylation_file_index_df.loc[
        methylation_file_index_df["sample_submitter_id"].isin(
            hm27_hm450_sample_ids
        )
    ]
    .groupby("project_id")["sample_submitter_id"]
    .nunique()
    .rename("paired_samples")
    .reset_index()
)

paired_aliquots_by_project_df = (
    methylation_file_index_df.loc[
        methylation_file_index_df["aliquot_uuid"].isin(
            hm27_hm450_aliquot_ids
        )
    ]
    .groupby("project_id")["aliquot_uuid"]
    .nunique()
    .rename("paired_aliquots")
    .reset_index()
)

hm27_hm450_pairing_by_project_df = (
    paired_cases_by_project_df
    .merge(
        paired_samples_by_project_df,
        on="project_id",
        how="outer",
    )
    .merge(
        paired_aliquots_by_project_df,
        on="project_id",
        how="outer",
    )
    .fillna(0)
)

count_columns = [
    "paired_cases",
    "paired_samples",
    "paired_aliquots",
]

hm27_hm450_pairing_by_project_df[count_columns] = (
    hm27_hm450_pairing_by_project_df[count_columns]
    .astype("int64")
)

hm27_hm450_pairing_by_project_df = (
    hm27_hm450_pairing_by_project_df
    .sort_values(
        [
            "paired_aliquots",
            "paired_cases",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

print("Cross-platform pairing by identifier level:")
display(cross_platform_pairing_df)

print("\nHM27–HM450 pairing by TCGA project:")
display(hm27_hm450_pairing_by_project_df)

Cross-platform pairing by identifier level:


,platform_pair,paired_cases,paired_samples,paired_aliquots
0,HM27 + HM450,215,213,194
1,HM27 + EPICv2,2,2,0
2,HM450 + EPICv2,51,51,0



HM27–HM450 pairing by TCGA project:


,project_id,paired_cases,paired_samples,paired_aliquots
0,TCGA-LAML,194,194,194
1,TCGA-LUAD,6,6,0
2,TCGA-GBM,5,5,0
3,TCGA-COAD,4,3,0
4,TCGA-KIRC,3,3,0
5,TCGA-BRCA,1,0,0
6,TCGA-READ,1,1,0
7,TCGA-UCEC,1,1,0


In [24]:
# =============================================================================
# Compare candidate methylation acquisition strategies
# =============================================================================

platform_strategy_definitions = {
    "All platforms": {
        "Illumina Human Methylation 27",
        "Illumina Human Methylation 450",
        "Illumina Methylation Epic v2",
    },
    "HM27 + HM450": {
        "Illumina Human Methylation 27",
        "Illumina Human Methylation 450",
    },
    "HM450 only": {
        "Illumina Human Methylation 450",
    },
    "HM27 only": {
        "Illumina Human Methylation 27",
    },
    "EPICv2 only": {
        "Illumina Methylation Epic v2",
    },
}

rna_cases_per_project = (
    rna_case_df
    .groupby("project_id")["case_submitter_id"]
    .nunique()
)

acquisition_strategy_records = []

for strategy_name, included_platforms in (
    platform_strategy_definitions.items()
):
    strategy_file_df = methylation_file_index_df.loc[
        methylation_file_index_df["platform"].isin(
            included_platforms
        )
    ].copy()

    strategy_case_ids = set(
        strategy_file_df["case_submitter_id"]
    )

    strategy_rna_overlap_ids = (
        strategy_case_ids & rna_case_ids
    )

    project_overlap_counts = (
        rna_case_df
        .assign(
            has_strategy_methylation=lambda df: (
                df["case_submitter_id"].isin(
                    strategy_case_ids
                )
            )
        )
        .groupby("project_id")[
            "has_strategy_methylation"
        ]
        .sum()
        .reindex(
            rna_cases_per_project.index,
            fill_value=0,
        )
    )

    project_rna_coverage_pct = (
        100
        * project_overlap_counts
        / rna_cases_per_project
    )

    acquisition_strategy_records.append(
        {
            "strategy": strategy_name,
            "n_files": strategy_file_df[
                "file_id"
            ].nunique(),
            "n_samples": strategy_file_df[
                "sample_submitter_id"
            ].nunique(),
            "n_cases": len(strategy_case_ids),
            "n_projects_with_methylation": (
                strategy_file_df["project_id"].nunique()
            ),
            "n_rna_overlap_cases": len(
                strategy_rna_overlap_ids
            ),
            "rna_case_coverage_pct": (
                100
                * len(strategy_rna_overlap_ids)
                / len(rna_case_ids)
            ),
            "n_projects_with_rna_overlap": int(
                (project_overlap_counts > 0).sum()
            ),
            "n_projects_without_rna_overlap": int(
                (project_overlap_counts == 0).sum()
            ),
            "minimum_project_rna_coverage_pct": (
                project_rna_coverage_pct.min()
            ),
            "median_project_rna_coverage_pct": (
                project_rna_coverage_pct.median()
            ),
            "download_size_gib": (
                strategy_file_df["file_size_bytes"].sum()
                / 1024**3
            ),
        }
    )

acquisition_strategy_comparison_df = pd.DataFrame(
    acquisition_strategy_records
)

print("Candidate methylation acquisition strategies:")

with pd.option_context(
    "display.max_columns",
    None,
):
    display(acquisition_strategy_comparison_df)

Candidate methylation acquisition strategies:


,strategy,n_files,n_samples,n_cases,n_projects_with_methylation,n_rna_overlap_cases,rna_case_coverage_pct,n_projects_with_rna_overlap,n_projects_without_rna_overlap,minimum_project_rna_coverage_pct,median_project_rna_coverage_pct,download_size_gib
0,All platforms,10950,10656,10610,33,10054,99.328196,33,0,80.985915,100.0,107.994131
1,HM27 + HM450,10897,10656,10610,33,10054,99.328196,33,0,80.985915,100.0,106.735678
2,HM450 only,8618,8598,8555,33,8376,82.750445,33,0,2.112676,100.0,105.103386
3,HM27 only,2279,2271,2270,12,1844,18.217744,12,21,0.000000,0.0,1.632291
4,EPICv2 only,53,53,53,1,52,0.513732,1,32,0.000000,0.0,1.258453


In [26]:
# =============================================================================
# Compare source-complete and RNA-overlap acquisition scopes
# =============================================================================

selected_analysis_platforms = {
    "Illumina Human Methylation 27",
    "Illumina Human Methylation 450",
}

hm27_hm450_file_df = methylation_file_index_df.loc[
    methylation_file_index_df["platform"].isin(
        selected_analysis_platforms
    )
].copy()

source_complete_mask = pd.Series(
    True,
    index=hm27_hm450_file_df.index,
)

rna_overlap_mask = (
    hm27_hm450_file_df["case_submitter_id"]
    .isin(rna_case_ids)
)

methylation_only_mask = ~rna_overlap_mask

acquisition_scope_masks = {
    "Source-complete HM27 + HM450": source_complete_mask,
    "RNA-overlap HM27 + HM450": rna_overlap_mask,
    "Methylation-only remainder": methylation_only_mask,
}

acquisition_scope_records = []

for scope_name, scope_mask in acquisition_scope_masks.items():
    scope_file_df = hm27_hm450_file_df.loc[
        scope_mask
    ]

    scope_case_ids = set(
        scope_file_df["case_submitter_id"]
    )

    acquisition_scope_records.append(
        {
            "scope": scope_name,
            "n_files": scope_file_df["file_id"].nunique(),
            "n_aliquots": scope_file_df[
                "aliquot_uuid"
            ].nunique(),
            "n_samples": scope_file_df[
                "sample_submitter_id"
            ].nunique(),
            "n_cases": len(scope_case_ids),
            "n_projects": scope_file_df[
                "project_id"
            ].nunique(),
            "n_rna_overlap_cases": len(
                scope_case_ids & rna_case_ids
            ),
            "download_size_gib": (
                scope_file_df["file_size_bytes"].sum()
                / 1024**3
            ),
        }
    )

acquisition_scope_comparison_df = pd.DataFrame(
    acquisition_scope_records
)

source_complete_size_gib = (
    acquisition_scope_comparison_df.loc[
        acquisition_scope_comparison_df["scope"]
        == "Source-complete HM27 + HM450",
        "download_size_gib",
    ].iloc[0]
)

rna_overlap_size_gib = (
    acquisition_scope_comparison_df.loc[
        acquisition_scope_comparison_df["scope"]
        == "RNA-overlap HM27 + HM450",
        "download_size_gib",
    ].iloc[0]
)

download_savings_gib = (
    source_complete_size_gib
    - rna_overlap_size_gib
)

download_savings_pct = (
    100
    * download_savings_gib
    / source_complete_size_gib
)

print("HM27/HM450 acquisition-scope comparison:")
display(acquisition_scope_comparison_df)

print(
    "\nPotential saving from RNA-overlap restriction:"
)
print(f"- Size:       {download_savings_gib:,.3f} GiB")
print(f"- Percentage: {download_savings_pct:,.2f}%")

HM27/HM450 acquisition-scope comparison:


,scope,n_files,n_aliquots,n_samples,n_cases,n_projects,n_rna_overlap_cases,download_size_gib
0,Source-complete HM27 + HM450,10897,10703,10656,10610,33,10054,106.735678
1,RNA-overlap HM27 + HM450,10289,10141,10099,10054,33,10054,104.246730
2,Methylation-only remainder,608,562,557,556,25,0,2.488948



Potential saving from RNA-overlap restriction:
- Size:       2.489 GiB
- Percentage: 2.33%


## Summary

This notebook assessed the metadata-level coverage of harmonized TCGA primary-tumor methylation beta-value files before defining the final acquisition cohort.

### Candidate all-platform cohort

The GDC query returned:

- 10,950 methylation beta-value files;
- 10,756 aliquots;
- 10,656 samples;
- 10,610 cases;
- 33 TCGA projects;
- 107.994 GiB of expected payload data.

The manifest, sample sheet, and metadata JSON contained identical file sets and fully concordant filenames, sizes, MD5 values, states, data categories, and data types. All files were open-access, released TXT outputs generated with `SeSAMe Methylation Beta Estimation`.

### Platform coverage

Three array platforms were represented:

- HM450: 8,618 files, 8,555 cases, 33 projects, and 105.103 GiB;
- HM27: 2,279 files, 2,270 cases, 12 projects, and 1.632 GiB;
- EPIC v2: 53 files, 53 cases, one project, and 1.258 GiB.

EPIC v2 contributed no exclusive cases. All 53 EPIC v2 cases were also represented by HM27 or HM450.

HM27 contributed 2,055 cases without HM450 coverage. These cases were strongly concentrated by project. In particular, an HM450-only strategy would retain only:

- 10 of 592 methylation cases in TCGA-OV;
- 140 of 422 in TCGA-GBM;
- 98 of 165 in TCGA-READ;
- 319 of 535 in TCGA-KIRC;
- 295 of 457 in TCGA-COAD;
- 784 of 1,097 in TCGA-BRCA.

Therefore, HM450-only acquisition would introduce substantial project- and lineage-dependent attrition.

### Cross-platform pairing

A total of 215 cases were represented by both HM27 and HM450:

- 213 were paired at the sample level;
- 194 were paired at the aliquot level.

All 194 same-aliquot HM27–HM450 pairs belonged to TCGA-LAML. These pairs may support a technical concordance assessment, but they do not provide a lineage-diverse basis for learning a global cross-platform correction applicable to solid tumors.

### RNA-seq overlap

The frozen RNA-seq cohort contained 10,122 cases.

The platform-union methylation cohort overlapped:

- 10,054 RNA-seq cases;
- 99.33% of the frozen RNA-seq cohort;
- all 33 TCGA projects.

HM450 alone overlapped 8,376 RNA-seq cases, corresponding to 82.75% of the RNA-seq cohort and 1,678 fewer multi-omic cases than the HM27/HM450 union.

The loss under HM450-only selection was strongly project-dependent. HM450 covered only 2.11% of RNA-seq cases in TCGA-OV and 28.17% in TCGA-GBM, despite providing complete or near-complete coverage in many other projects.

### Acquisition-scope assessment

Two HM27/HM450 acquisition scopes were compared:

- source-complete: 10,897 files, 10,610 cases, and 106.736 GiB;
- restricted to RNA-seq overlap: 10,289 files, 10,054 cases, and 104.247 GiB.

Restricting acquisition to the current RNA-seq overlap would save only 2.489 GiB, or 2.33% of the source-complete HM27/HM450 download. This limited saving does not justify coupling the methylation source cohort prematurely to the RNA-seq cohort.

### Acquisition decision

The selected acquisition strategy is:

- retain all source-complete HM27 and HM450 files;
- exclude EPIC v2 files;
- retain all 33 TCGA projects;
- defer sample and aliquot selection until methylation quality control;
- preserve platform identity throughout downstream processing.

The final acquisition cohort will therefore contain:

- 10,897 files;
- 10,656 samples;
- 10,610 cases;
- 33 TCGA projects;
- 106.736 GiB of expected payload data.

### Downstream methodological constraints

HM27 and HM450 beta-value matrices must not be pooled naïvely. Platform identity must remain explicit during quality control and downstream evaluation.

The eventual analytical strategy may include platform-stratified analyses, shared-probe or gene-level representations, and full-resolution HM450 sensitivity analyses. The exact integration strategy will be determined after probe-level and sample-level quality assessment.

No methylation payloads were downloaded or parsed in this notebook. No final multi-omic cohort was constructed.

The selected HM27/HM450 file-level cohort will be frozen in the subsequent TCGA methylation cohort-freeze notebook.